# Module 0.2: Environment Setup for HMM Development

## Learning Objectives
By completing this notebook, you will:
1. Set up a Python environment for HMM development
2. Install and verify essential libraries (NumPy, SciPy, hmmlearn, pomegranate)
3. Test basic HMM functionality with built-in libraries
4. Configure visualization tools for regime analysis
5. Validate your setup with a complete HMM workflow

## Prerequisites
- Python 3.8+ installed
- Basic command line familiarity
- pip or conda package manager

## Estimated Time: 30 minutes

---

## 1. Required Libraries

We'll use the following libraries throughout this course:

### Core Scientific Computing
- **NumPy**: Array operations and linear algebra
- **SciPy**: Statistical distributions and optimization
- **Pandas**: Data manipulation and time series

### HMM-Specific Libraries
- **hmmlearn**: Standard HMM implementation
- **pomegranate**: Flexible probabilistic models

### Visualization
- **Matplotlib**: Basic plotting
- **Seaborn**: Statistical visualizations
- **Plotly** (optional): Interactive plots

## 2. Installation Commands

Run these commands in your terminal (not in this notebook):

```bash
# Using pip
pip install numpy scipy pandas matplotlib seaborn
pip install hmmlearn scikit-learn
pip install jupyter jupyterlab

# Optional: pomegranate (may require compilation)
pip install pomegranate

# Optional: interactive plotting
pip install plotly
```

Or using conda:
```bash
conda install numpy scipy pandas matplotlib seaborn scikit-learn jupyter
conda install -c conda-forge hmmlearn
```

## 3. Verify Installation

In [ ]:
# Purpose: Verify all required libraries are installed
# Key Concept: Check versions and import functionality

import sys
print(f"Python version: {sys.version}\n")

# Test imports and show versions
libraries = [
    ('numpy', 'np'),
    ('scipy', None),
    ('pandas', 'pd'),
    ('matplotlib', None),
    ('seaborn', 'sns'),
    ('sklearn', None),
    ('hmmlearn', None),
]

print("Library Versions:")
print("="*60)

for lib_name, alias in libraries:
    try:
        if alias:
            exec(f"import {lib_name} as {alias}")
            exec(f"version = {alias}.__version__")
        else:
            exec(f"import {lib_name}")
            exec(f"version = {lib_name}.__version__")
        
        print(f"✓ {lib_name:12s} : {version}")
    except ImportError:
        print(f"✗ {lib_name:12s} : NOT INSTALLED")
    except AttributeError:
        print(f"✓ {lib_name:12s} : installed (version unknown)")

# Test optional libraries
print("\nOptional Libraries:")
print("="*60)

optional = ['plotly', 'pomegranate']
for lib in optional:
    try:
        exec(f"import {lib}")
        try:
            exec(f"version = {lib}.__version__")
            print(f"✓ {lib:12s} : {version}")
        except:
            print(f"✓ {lib:12s} : installed")
    except ImportError:
        print(f"- {lib:12s} : not installed (optional)")

## 4. Test Basic Functionality

In [ ]:
# Purpose: Test that NumPy and SciPy work correctly
# Key Concept: Verify numerical operations and distributions

import numpy as np
from scipy import stats

# Test NumPy
print("NumPy Test:")
print("="*60)

# Create transition matrix
A = np.array([[0.7, 0.3], [0.4, 0.6]])
print(f"Transition matrix:\n{A}")
print(f"Row sums: {A.sum(axis=1)}")
print(f"Eigenvalues: {np.linalg.eig(A.T)[0]}")

# Test SciPy
print("\nSciPy Test:")
print("="*60)

# Gaussian distribution
rv = stats.norm(loc=0.001, scale=0.01)
samples = rv.rvs(size=1000, random_state=42)
print(f"Generated {len(samples)} samples from N(0.001, 0.01²)")
print(f"Sample mean: {np.mean(samples):.6f}")
print(f"Sample std:  {np.std(samples):.6f}")

print("\n✅ NumPy and SciPy are working correctly!")

## 5. Test hmmlearn Library

In [ ]:
# Purpose: Test hmmlearn with a simple example
# Key Concept: Verify Gaussian HMM functionality

from hmmlearn import hmm
import numpy as np

print("Testing hmmlearn:")
print("="*60)

# Generate synthetic data
np.random.seed(42)

# True parameters (2-state HMM)
true_means = np.array([[0.001], [-0.002]])  # Bull vs Bear
true_vars = np.array([[0.01**2], [0.02**2]])

# Simulate returns
n_samples = 500
true_states = np.random.choice(2, size=n_samples, p=[0.7, 0.3])
returns = np.array([
    np.random.normal(true_means[s, 0], np.sqrt(true_vars[s, 0]))
    for s in true_states
]).reshape(-1, 1)

print(f"Generated {n_samples} synthetic returns\n")

# Fit Gaussian HMM
model = hmm.GaussianHMM(
    n_components=2,
    covariance_type="full",
    n_iter=100,
    random_state=42
)

model.fit(returns)

print("Learned Parameters:")
print(f"Means:\n{model.means_}")
print(f"\nCovariances:\n{model.covars_}")
print(f"\nTransition matrix:\n{model.transmat_}")

# Decode states
predicted_states = model.predict(returns)

# Compute accuracy (account for label permutation)
accuracy = max(
    np.mean(predicted_states == true_states),
    np.mean(predicted_states == 1 - true_states)
)

print(f"\nDecoding accuracy: {accuracy:.1%}")
print("\n✅ hmmlearn is working correctly!")

## 6. Test Visualization

In [ ]:
# Purpose: Verify plotting functionality
# Key Concept: Ensure Matplotlib and Seaborn work in notebooks

import matplotlib.pyplot as plt
import seaborn as sns

# Configure
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("Testing Visualization:")
print("="*60 + "\n")

# Create test plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Returns
ax1 = axes[0]
ax1.plot(returns, 'k-', alpha=0.6, linewidth=0.8)
ax1.set_ylabel('Return', fontsize=12)
ax1.set_title('Simulated Returns', fontsize=14)
ax1.axhline(0, color='red', linestyle='--', alpha=0.3)
ax1.grid(True, alpha=0.3)

# Plot 2: States
ax2 = axes[1]
# Align predicted states with true states
if np.mean(predicted_states == true_states) < 0.5:
    predicted_states = 1 - predicted_states

ax2.fill_between(range(n_samples), predicted_states, alpha=0.6, 
                 label='Predicted', step='mid')
ax2.set_xlabel('Time', fontsize=12)
ax2.set_ylabel('State', fontsize=12)
ax2.set_title('Decoded States', fontsize=14)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Bull', 'Bear'])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualization is working correctly!")

## 7. Complete Workflow Test

Let's run a complete HMM workflow to ensure everything works together.

In [ ]:
# Purpose: End-to-end HMM workflow validation
# Key Concept: Simulate → Train → Decode → Visualize

print("Complete HMM Workflow Test:")
print("="*60 + "\n")

# Step 1: Simulate market data
print("Step 1: Simulating market data...")
n_days = 1000
market_states = [0]  # Start in Bull
market_returns = []

# Transition probabilities
A_market = np.array([[0.95, 0.05], [0.10, 0.90]])
means_market = [0.0008, -0.0015]
stds_market = [0.010, 0.025]

for _ in range(n_days - 1):
    current_state = market_states[-1]
    next_state = np.random.choice(2, p=A_market[current_state])
    market_states.append(next_state)

for state in market_states:
    ret = np.random.normal(means_market[state], stds_market[state])
    market_returns.append(ret)

market_returns = np.array(market_returns).reshape(-1, 1)
market_states = np.array(market_states)
print(f"✓ Generated {n_days} days of returns\n")

# Step 2: Train HMM
print("Step 2: Training HMM...")
model_test = hmm.GaussianHMM(
    n_components=2,
    covariance_type="full",
    n_iter=50,
    random_state=123,
    verbose=False
)
model_test.fit(market_returns)
print(f"✓ Converged in {model_test.monitor_.iter} iterations\n")

# Step 3: Decode regimes
print("Step 3: Decoding regimes...")
decoded = model_test.predict(market_returns)
accuracy_final = max(
    np.mean(decoded == market_states),
    np.mean(decoded == 1 - market_states)
)
print(f"✓ Decoding accuracy: {accuracy_final:.1%}\n")

# Step 4: Analyze results
print("Step 4: Analyzing results...")
log_prob = model_test.score(market_returns)
print(f"Log-likelihood: {log_prob:.2f}")
print(f"Per-observation: {log_prob/n_days:.4f}\n")

print("Learned parameters:")
for i in range(2):
    state_name = 'Bull' if model_test.means_[i, 0] > 0 else 'Bear'
    print(f"  State {i} ({state_name}):")
    print(f"    Mean: {model_test.means_[i, 0]:.6f}")
    print(f"    Std:  {np.sqrt(model_test.covars_[i, 0, 0]):.6f}")

print("\n" + "="*60)
print("✅ ALL TESTS PASSED - Environment is ready!")
print("="*60)
print("\nYou can now proceed with the course modules.")

## 8. Troubleshooting Common Issues

### Issue 1: Import Errors

**Problem:** `ModuleNotFoundError: No module named 'hmmlearn'`

**Solution:**
```bash
pip install --upgrade hmmlearn
```

### Issue 2: Jupyter Not Finding Packages

**Problem:** Packages work in terminal but not in Jupyter

**Solution:**
```bash
# Install in Jupyter's Python environment
python -m pip install hmmlearn numpy scipy

# Or ensure Jupyter uses correct kernel
python -m ipykernel install --user --name=hmm-env
```

### Issue 3: Plotting Not Working

**Problem:** Plots don't appear in notebook

**Solution:**
```python
# Add this at the start of your notebook
%matplotlib inline
import matplotlib.pyplot as plt
```

### Issue 4: Performance Issues

**Problem:** Training is very slow

**Solution:**
```python
# Use fewer iterations for testing
model = hmm.GaussianHMM(n_components=2, n_iter=50)  # Instead of 100+

# Check if NumPy is using optimized BLAS
import numpy as np
np.show_config()  # Should show MKL or OpenBLAS
```

## Summary

### Environment Checklist

- ✓ Python 3.8+
- ✓ NumPy for array operations
- ✓ SciPy for statistical functions
- ✓ Pandas for data manipulation
- ✓ hmmlearn for HMM implementation
- ✓ Matplotlib/Seaborn for visualization
- ✓ Jupyter for interactive development

### What's Next

Now that your environment is set up, you're ready to:

1. **Module 1**: Learn HMM framework and terminology
2. **Module 2**: Implement core algorithms (Forward-Backward, Viterbi, Baum-Welch)
3. **Module 3**: Apply Gaussian HMMs to financial data
4. **Module 4**: Build regime-based trading strategies

### Additional Resources

- [NumPy Documentation](https://numpy.org/doc/)
- [SciPy Documentation](https://docs.scipy.org/doc/scipy/)
- [hmmlearn Documentation](https://hmmlearn.readthedocs.io/)
- [Jupyter Notebook Guide](https://jupyter-notebook.readthedocs.io/)

---

**Next Notebook:** Module 1 - HMM Framework and Definitions